In [1]:
import numpy as np
import torch, torchaudio
import os
from torchinfo import summary
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm.auto import tqdm

/opt/miniconda3/envs/deeplearning/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
!mkdir -p ~/.kaggle

with open("~/.kaggle/kaggle.json", "w+") as f:
    f.write('{"username":"barnabsepres","key":"1a71ef4d59648c853c5f0a24fc2a2aea"}')
    # Put your kaggle username & key here

#!chmod 600 /root/.kaggle/kaggle.json

FileNotFoundError: [Errno 2] No such file or directory: '~/.kaggle/kaggle.json'

In [14]:
#!kaggle competitions download -c 11785-hw1p2-f24

!unzip -qo /content/11785-hw1p2-f24.zip -d '/content'

unzip:  cannot find or open /content/11785-hw1p2-f24.zip, /content/11785-hw1p2-f24.zip.zip or /content/11785-hw1p2-f24.zip.ZIP.


In [2]:
PHONEMES = [
            '[SIL]',   'AA',    'AE',    'AH',    'AO',    'AW',    'AY',
            'B',     'CH',    'D',     'DH',    'EH',    'ER',    'EY',
            'F',     'G',     'HH',    'IH',    'IY',    'JH',    'K',
            'L',     'M',     'N',     'NG',    'OW',    'OY',    'P',
            'R',     'S',     'SH',    'T',     'TH',    'UH',    'UW',
            'V',     'W',     'Y',     'Z',     'ZH',    '[SOS]', '[EOS]']

In [3]:
#it is a class for loading the audiodataset (it will be called i think during the training process for the epochs)

class AudioDataset(torch.utils.data.Dataset):
    #in the init method i input the 
    # root: folder my data is in, 
    # the phonemes list:  is already defined, 
    # also context : maybe how many data i would like to retrieve
    # partition: the suffix to get data from a specific folder
    def __init__(self, root, phonemes = PHONEMES, context = 0, partition = "train-clean-100"):
        self.context, self.phonemes = context, phonemes

        #setting the paths for the mel spectograms (is this an image of a spectogram, or a file or ones and zeros representing data of the spectogram?)    
        self.mfcc_dir = f"{root}/{partition}/mfcc"
        #path for transcripts, in my case phonemes
        self.transcript_dir = f"{root}/{partition}/transcript"

        #get the names of all mel spectogram and phoneme transcript files
        mfcc_names = sorted(os.listdir(self.mfcc_dir))
        transcript_names = sorted(os.listdir(self.transcript_dir))

        #make sure that they are the same lenght, aka there is exactly one phoneme for one spectogram
        assert len(mfcc_names) == len(transcript_names)

        #get empty lists for the spectograms and transcripts itselves
        self.mfccs, self.transcripts = [], []

        for i in range(len(mfcc_names)):
            #loop through the range of number of data and get the first mel spectogram
            mfcc = np.load(f'{self.mfcc_dir}/{mfcc_names[i]}')

            #cepstral normalization for mfcc to remove any channel effects, i dont quite understand how can I take the mean and std of a mel spectogram, what kind of data is that? 
            mfcc_mean = np.mean(mfcc, axis=0)
            mfcc_std = np.std(mfcc, axis=0)
            mfcc = (mfcc-mfcc_mean) / (mfcc_std + 1e-8)

            #load the corersponding transcript and remove SOF AND EOF
            transcript = np.load(f'{self.transcript_dir}/{transcript_names[i]}')[1:-1]

            #append to the corresponding lists
            self.mfccs.append(mfcc)
            self.transcripts.append(transcript)
        
        #concatenate the list so that now it is a multi dimensional vector, each mel spectogram is a vector and self.mfccs contains a lot of these vectors
        self.mfccs = np.concatenate(self.mfccs, axis=0)

        #same as with mel spectograms
        self.transcripts = np.concatenate(self.transcripts, axis=0)

        #get how many spectograms/data there is
        self.length = len(self.mfccs)

        #add some kind of padding to it? 
        self.mfccs = np.pad(self.mfccs, ((self.context, self.context), (0, 0)), mode="constant")

        #representing phonemes with indexes so that y will be numerical
        phoneme_to_index = {phoneme: idx for idx, phoneme in enumerate(self.phonemes)}

        self.transcripts = np.array([phoneme_to_index[phoneme] for phoneme in self.transcripts])

    #function to return the amount of data
    def __len__(self):
        return self.length

    #function for getting data? what is index here? and context? 
    #ah i think i know, context is some extra frames before and after the actual frame so that we will have some context 
    def __getitem__(self, ind):
        frames = self.mfccs[ind:ind + 2 * self.context + 1]
        frames = frames.flatten() #make them a single array instead of as many dimensions as frames

        #convert frames and phonemes to pytorch tensors
        frames = torch.FloatTensor(frames) #convert to tensors
        phonemes = torch.tensor(self.transcripts[ind])

        return frames, phonemes

In [4]:
class AudioTestDataset(torch.utils.data.Dataset):
    def __init__(self, root, context = 0, partition = "test-clean"):
        self.root = root
        self.mfcc_dir = f'{root}/{partition}/mfcc'
        self.mfcc_names = sorted(os.listdir(self.mfcc_dir))

        self.length = len(self.mfcc_names)
        self.mfccs = []

        for i in range(len(self.mfcc_names)):
            mfcc = np.load(f'{self.mfcc_dir}/{self.mfcc_names[i]}')

            #normalization
            mfcc_mean = np.mean(mfcc, axis=0)
            mfcc_std = np.std(mfcc, axis=0)
            mfcc = (mfcc-mfcc_mean) / (mfcc_std + 1e-8)

            self.mfccs.append(mfcc)

        def __len__(self):
            return self.length

        def __getitem__(self, ind):
            frames = self.mfccs[ind: ind + 2 * self.context + 1]
            frames = frames.flatten()
            frames = torch.FloatTensor(frames)
            return frames

In [5]:
#params config
config = {
    "epochs": 104,
    "batch_size": 2048,
    "context": 35,
    'init_lr': 1e-3,
'scheduler': {
        'T_0': 7,
        'T_mult': 2,
        'eta_min': 1e-5,
        'type': 'CosineAnnealingWarnRestartsLR'
    },

    'weight_decay': 1e-3,
}

root = "./content/11785-f24-hw1p2"

In [6]:
#create datasets

#i create instances of the AudioDataset with the correct parameters
train_data = AudioDataset(root, context=config['context'], partition="train-clean-100")

val_data = AudioDataset(root, context=config['context'], partition="dev-clean")

test_data = AudioTestDataset(root, context=config['context'], partition="test-clean")


In [7]:
#create a train loader, which is an instance of torch DataLoader class? 
#it uses the train_data aka AudioDataset class as dataset, i guess the __getitem__() function which gives back batch size amount of data
train_loader = torch.utils.data.DataLoader(
    dataset = train_data,
    #num_workers = 4, 
    batch_size = config['batch_size'], 
    pin_memory = False, 
    shuffle=True
)

val_loader = torch.utils.data.DataLoader(
    dataset = val_data,
    #num_workers = 4, 
    batch_size = config['batch_size'], 
    pin_memory = True, 
    shuffle=False
)

test_loader = torch.utils.data.DataLoader(
    dataset = test_data,
    #num_workers = 4, 
    batch_size = config['batch_size'], 
    pin_memory = False, 
    shuffle=False
)

In [8]:
#test the dataloader
for i, data in enumerate(train_loader):
    frames, phoneme = data
    print(frames.shape, phoneme.shape)
    break

torch.Size([2048, 1988]) torch.Size([2048])


## Define network architecture

In [9]:
#neural network
class Network(torch.nn.Module):
    """
    Multi layer perceptron neural network
    """
    #this initializes the input size the output size and also other functions from the torch.nn.Module with super? It was a long time ago since i have studied nested classes in python
    def __init__(self, input_size, output_size):
        super(Network, self).__init__()

        #make the model variable a sequential, so i can call it at once
        self.model = torch.nn.Sequential(
            #make the data from input size to 2048 dimensions
            torch.nn.Linear(input_size, 2048), 
            #what is batch normalization here? This model receives only a few frames to my understanding, does it normalize them? 
            #arent they already normalized when they are loaded? refer to my previous code about AudioDataset class
            #everything else is clear here
            torch.nn.BatchNorm1d(2048),
            torch.nn.ReLU(),

            torch.nn.Linear(2048, 1024),
            torch.nn.BatchNorm1d(1024),            
            torch.nn.GELU(),

            torch.nn.Linear(1024, 512), 
            torch.nn.BatchNorm1d(512),
            torch.nn.GELU(),

            torch.nn.Linear(512, 256),
            torch.nn.GELU(),

            torch.nn.Linear(256, output_size)
        )

    def forward(self, x):
        out = self.model(x)
        return out

## Define model, loss function, optimizer

In [10]:
INPUT_SIZE = (2*config['context'] + 1)*28
model = Network(INPUT_SIZE, len(train_data.phonemes))
#summary(model, (INPUT_SIZE,))

In [11]:
#define criterior, or loss function, here i use cross entropy loss for multiclass classification
#it is used to calculate the loss, so for a batch how much the original and the predicted y values differ
criterion = torch.nn.CrossEntropyLoss()


#use adam optimizer
#optimizer: optimizer decides that which direction should the gradient change? 
# it uses a learning rate which is used for how "fast" the gradient changes
# what is weight decay here? I have only used sgd before
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=config['init_lr'],
                                 weight_decay=config['weight_decay'])

# i am not sure what this scheduler or cosineannealing is and how it is used
scheduler = CosineAnnealingWarmRestarts(optimizer, 
                              T_0 = config['scheduler']['T_0'],
                                eta_min=config['scheduler']['eta_min'],
                                  T_mult = config['scheduler']['T_mult'])

## Training and validation fucntions

In [13]:
#training and validation functions

def train(model, dataloader, optimizer, criterion):
    """
    Training function, takes model, dataloader, optimizer and criterion as inputs
    """

    #this sets the model in train? Is it derived from the torch.nn.Module class?
    model.train()
    #define training loss and training accuracy
    tloss, tacc = 0, 0
    #batch bar, do not care about it
    batch_bar   = tqdm(total=len(dataloader), dynamic_ncols=True, leave=False, position=0, desc='Train')
    
    #a for loop which runs through all data in the dataloader, gets the index, the frames and the corresponding phonemes
    for i, (frames, phonemes) in enumerate(dataloader):
        #delete all gradients from the optimizer to start fresh and do not accumulate gradients
        optimizer.zero_grad()

        #get logits, aka the predicted phoneme from the neural net for the current frames
        logits = model(frames)

        #get the loss value, the cross entropy loss difference between the predicted the and real phonemes 
        loss = criterion(logits, phonemes)

        #backpropagate based on the loss
        loss.backward()

        #take an optimizer step: move the gradient somewhere, towards the functions minimum
        optimizer.step()

        #append the loss from this iteration to the total training loss
        tloss += loss.item()

        #update the accuracy
        tacc += torch.sum(torch.argmax(logits, dim=1) == phonemes).item()/logits.shape[0]

        batch_bar.set_postfix(loss="{:.04f}".format(float(tloss / (i + 1))),
                              acc="{:.04f}%".format(float(tacc*100 / (i + 1))))
        batch_bar.update()
        #delete frames, phonemes, logits? i dont get this syntax
        del frames, phonemes, logits

    batch_bar.close()
    tloss /= len(train_loader)
    tacc /= len(train_loader)
    return tloss, tacc

#evalute the model
def eval(model, dataloader):
    """
    Eval function, gets the model and the eval dataloader
    """
    #set the model into eval function?
    model.eval()
    vloss, vacc = 0, 0
    batch_bar = tqdm(total=len(val_loader),
                      dynamic_ncols=True,
                        position=0,
                          leave=False,
                          desc='Val')
    
    #get the index, frames and phonemes from the dataloader
    for i, (frames, phonemes) in enumerate(dataloader):

        #i am not sure what is this mode
        with torch.inference_mode():
            #get the predictions again and compare them to y
            logits = model(frames)
            loss = criterion(logits, phonemes)
        
        #update validation loss and validation accuracy
        vloss += loss.item()
        vacc += torch.sum(
            torch.argmax(logits, dim=1) == phonemes
        ).item() / logits.shape[0]

        batch_bar.set_postfix(loss="{:.04f}".format(float(vloss / (i + 1))),
                              acc="{:.04f}%".format(float(vacc*100 / (i + 1))))
        batch_bar.update()
        del frames, phonemes, logits
    
    batch_bar.close()
    vloss /= len(val_loader)
    vacc /=len(val_loader)

    return vloss, vacc

In [14]:
#train the model

#training it on 5 epochs, i could use epochs from the congfig but do not care about it right now
for epoch in range(5):
    print("\nEpoch {}/{}".format(epoch+1, config['epochs']))

    #getting the curren learning rate (it is good to have since it dynamically changes due to scheduler)
    curr_lr = float(optimizer.param_groups[0]['lr'])
    #use the train function to get training loss and training accuracy
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    #validate using eval function
    val_loss, val_acc = eval(model, val_loader)

    print("\tTrain Acc {:.04f}%\tTrain Loss {:.04f}\t Learning Rate {:.07f}".format(train_acc*100, train_loss, curr_lr))
    print("\tVal Acc {:.04f}%\tVal Loss {:.04f}".format(val_acc*100, val_loss))

    #take a step with the scheduler
    scheduler.step()

    #questions: if I was to use some other optimizer without the scheduler, i can just delete this line and use the other scjheduler inside the batch for loops, like optimnizer.step(), right?


Epoch 1/104


Val:   0%|          | 0/942 [00:00<?, ?it/s]                                             /opt/miniconda3/envs/deeplearning/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


	Train Acc 78.4406%	Train Loss 0.6590	 Learning Rate 0.0010000
	Val Acc 79.0138%	Val Loss 0.6401

Epoch 2/104


	Train Acc 84.1013%	Train Loss 0.4703	 Learning Rate 0.0009510
	Val Acc 79.7994%	Val Loss 0.6295

Epoch 3/104


	Train Acc 86.2403%	Train Loss 0.4008	 Learning Rate 0.0008136
	Val Acc 79.7804%	Val Loss 0.6476

Epoch 4/104


	Train Acc 87.8341%	Train Loss 0.3503	 Learning Rate 0.0006151
	Val Acc 79.6401%	Val Loss 0.6825

Epoch 5/104


	Train Acc 89.2248%	Train Loss 0.3078	 Learning Rate 0.0003949
	Val Acc 79.4147%	Val Loss 0.7291
